# 02 — Fitur Double-Dummy Solver (DDS)

**Catatan scope**: DDS awalnya di luar ruang lingkup skripsi (lihat
`CLAUDE.md`), disetujui masuk 2026-07-15 sebagai fitur TAMBAHAN opsional
di `experiments/` — bukan pengganti 164 fitur kanonik — setelah temuan
notebook 01 (batas konsistensi bidding manusia sendiri ~37.6%).

DDS (`endplay`, wrapper Python untuk DDS-nya Bo Haglund) menghitung,
untuk setiap papan, **jumlah trick optimal** yang bisa diambil tiap sisi
di tiap strain dengan asumsi permainan sempurna & semua kartu terbuka —
dan **kontrak par** (kontrak teoretis terbaik, plus skornya). Ini fitur
yang jauh lebih dekat ke "seharusnya kontrak apa" dibanding proxy seperti
HCP/LTC/fit — tapi TIDAK menghapus ambiguitas gaya bidding manusia (par
≠ kontrak yang benar-benar dibid, itu sebabnya manusia sendiri cuma
37.6% konsisten satu sama lain, bukan 100%).

Fitur baru (18 kolom):

| Fitur | Keterangan |
|---|---|
| `ns_dd_S/H/D/C/N`, `ew_dd_S/H/D/C/N` | Trick DD per strain per partnership (10 kolom) |
| `dd_par_level` | Level kontrak par (1-7) |
| `dd_par_denom_S/H/D/C/N` | One-hot strain kontrak par (5 kolom) |
| `dd_par_declarer_is_ns` | 1 jika par dibidik NS, 0 jika EW |
| `dd_par_score` | Skor par (perspektif NS, standar scoring bridge) |

Dihitung sekali untuk seluruh `data/raw/` (~10.223 papan, cache di
`outputs/dds_cache.csv`, ~40 menit), lalu di-join ke split kanonik
(group-aware) lewat `(_source_file, _room, _board_number)` — pola sama
seperti fitur tambahan 2026-07-09.


In [1]:
import sys
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

PROJECT_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing.dataset_builder import load_splits
from src.evaluation.metrics import evaluate, print_summary, compare_models

OUT_DIR = PROJECT_ROOT / "experiments" / "2026-07-15" / "outputs" / "dds_features"
OUT_DIR.mkdir(parents=True, exist_ok=True)

DDS_CACHE_PATH = PROJECT_ROOT / "experiments" / "2026-07-15" / "outputs" / "dds_cache.csv"
df_dds = pd.read_csv(DDS_CACHE_PATH)
print(f"DDS cache: {df_dds.shape}")
df_dds.head()


DDS cache: (11225, 17)


,_source_file,_room,_board_number,ns_dd_S,ew_dd_S,ns_dd_H,ew_dd_H,ns_dd_D,ew_dd_D,ns_dd_C,ew_dd_C,ns_dd_N,ew_dd_N,dd_par_level,dd_par_denom,dd_par_declarer_is_ns,dd_par_score
0,85168.lin,open,13,3,10,3,10,6,7,3,10,4,9,4,0,0,-620
1,85168.lin,closed,13,3,10,3,10,6,7,3,10,4,9,4,0,0,-620
2,85168.lin,open,14,10,3,5,8,7,5,2,11,8,4,5,0,1,-100
3,85168.lin,closed,14,10,3,5,8,7,5,2,11,8,4,5,0,1,-100
4,85168.lin,open,15,6,7,4,9,4,9,6,7,4,8,1,1,0,-140


## 1. Turunkan fitur tambahan dari DDS mentah (one-hot strain par, dsb.)

In [2]:
STRAINS = ["S", "H", "D", "C", "N"]

df_dds = df_dds.copy()
for s in STRAINS:
    df_dds[f"dd_par_denom_{s}"] = (df_dds["dd_par_denom"] == STRAINS.index(s)).astype(int)
df_dds = df_dds.drop(columns=["dd_par_denom"])

dds_feature_cols = [c for c in df_dds.columns if c not in ("_source_file", "_room", "_board_number")]
print(f"DDS feature columns ({len(dds_feature_cols)}): {dds_feature_cols}")


DDS feature columns (18): ['ns_dd_S', 'ew_dd_S', 'ns_dd_H', 'ew_dd_H', 'ns_dd_D', 'ew_dd_D', 'ns_dd_C', 'ew_dd_C', 'ns_dd_N', 'ew_dd_N', 'dd_par_level', 'dd_par_declarer_is_ns', 'dd_par_score', 'dd_par_denom_S', 'dd_par_denom_H', 'dd_par_denom_D', 'dd_par_denom_C', 'dd_par_denom_N']


## 2. Join ke split kanonik (group-aware, sama seperti pipeline utama)

In [3]:
df_train, df_val, df_test, feature_cols, le = load_splits(PROJECT_ROOT / "data" / "processed")

JOIN_KEYS = ["_source_file", "_room", "_board_number"]


def augment(df_split: pd.DataFrame) -> pd.DataFrame:
    before = len(df_split)
    merged = df_split.merge(df_dds, on=JOIN_KEYS, how="left", validate="one_to_one")
    assert len(merged) == before, "Join changed row count"
    n_missing = merged[dds_feature_cols].isnull().sum().sum()
    assert n_missing == 0, f"{n_missing} DDS values missing after join"
    return merged


df_train_aug = augment(df_train)
df_val_aug = augment(df_val)
df_test_aug = augment(df_test)

all_feature_cols = feature_cols + dds_feature_cols
print(f"Total fitur: {len(all_feature_cols)} (164 kanonik + {len(dds_feature_cols)} DDS)")

X_train_164 = df_train_aug[feature_cols].values
X_val_164 = df_val_aug[feature_cols].values
X_train_182 = df_train_aug[all_feature_cols].values
X_val_182 = df_val_aug[all_feature_cols].values
y_train = df_train_aug["label"].values
y_val = df_val_aug["label"].values

sw_train = compute_sample_weight("balanced", y_train)


Total fitur: 182 (164 kanonik + 18 DDS)


## 3. Bandingkan: baseline (164) vs +DDS (182) vs +DDS+sample_weight+DART (kombinasi terbaik 2026-07-09/10)

In [4]:
BASE_XGB = dict(
    n_estimators=300, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, random_state=42,
    n_jobs=-1, eval_metric="mlogloss", verbosity=0,
)
DART_PARAMS = dict(booster="dart", rate_drop=0.1, skip_drop=0.5, normalize_type="tree")

results = []

# A: baseline, 164 fitur
t0 = time.time()
clf = XGBClassifier(**BASE_XGB)
clf.fit(X_train_164, y_train)
res = evaluate(y_val, clf.predict(X_val_164), clf.predict_proba(X_val_164), le,
                model_name="A: XGBoost baseline (164 fitur)")
res["fit_seconds"] = round(time.time() - t0, 1)
results.append(res)
print_summary(res)

# B: +DDS, 182 fitur, XGBoost polos
t0 = time.time()
clf = XGBClassifier(**BASE_XGB)
clf.fit(X_train_182, y_train)
res = evaluate(y_val, clf.predict(X_val_182), clf.predict_proba(X_val_182), le,
                model_name="B: XGBoost +DDS (182 fitur)")
res["fit_seconds"] = round(time.time() - t0, 1)
results.append(res)
print_summary(res)

# C: +DDS + sample_weight + DART (kombinasi terbaik sebelumnya)
t0 = time.time()
clf = XGBClassifier(**{**BASE_XGB, **DART_PARAMS})
clf.fit(X_train_182, y_train, sample_weight=sw_train)
res = evaluate(y_val, clf.predict(X_val_182), clf.predict_proba(X_val_182), le,
                model_name="C: XGBoost +DDS +sample_weight +dart (182 fitur)")
res["fit_seconds"] = round(time.time() - t0, 1)
results.append(res)
print_summary(res)

# D (referensi): 164 fitur + sample_weight + DART, TANPA DDS (kandidat terbaik sebelum hari ini)
t0 = time.time()
clf = XGBClassifier(**{**BASE_XGB, **DART_PARAMS})
clf.fit(X_train_164, y_train, sample_weight=sw_train)
res = evaluate(y_val, clf.predict(X_val_164), clf.predict_proba(X_val_164), le,
                model_name="D (REF): XGBoost +sample_weight +dart, TANPA DDS (164 fitur)")
res["fit_seconds"] = round(time.time() - t0, 1)
results.append(res)
print_summary(res)



  A: XGBoost baseline (164 fitur)
  Accuracy          : 0.5127
  Precision (macro) : 0.3335
  Recall (macro)    : 0.2515
  F1 (macro)        : 0.2685
  F1 (weighted)     : 0.4685
  Top-3 Accuracy    : 0.7508
  Top-5 Accuracy    : 0.8480



  B: XGBoost +DDS (182 fitur)
  Accuracy          : 0.5114
  Precision (macro) : 0.3370
  Recall (macro)    : 0.2600
  F1 (macro)        : 0.2786
  F1 (weighted)     : 0.4718
  Top-3 Accuracy    : 0.7515
  Top-5 Accuracy    : 0.8461



  C: XGBoost +DDS +sample_weight +dart (182 fitur)
  Accuracy          : 0.4847
  Precision (macro) : 0.3101
  Recall (macro)    : 0.3433
  F1 (macro)        : 0.3154
  F1 (weighted)     : 0.4914
  Top-3 Accuracy    : 0.7365
  Top-5 Accuracy    : 0.8330



  D (REF): XGBoost +sample_weight +dart, TANPA DDS (164 fitur)
  Accuracy          : 0.4918
  Precision (macro) : 0.3156
  Recall (macro)    : 0.3455
  F1 (macro)        : 0.3217
  F1 (weighted)     : 0.4983
  Top-3 Accuracy    : 0.7482
  Top-5 Accuracy    : 0.8330


## 4. Feature importance — di mana ranking fitur DDS?

In [5]:
clf_b = XGBClassifier(**BASE_XGB)
clf_b.fit(X_train_182, y_train)
importances = pd.Series(clf_b.feature_importances_, index=all_feature_cols).sort_values(ascending=False)
ranking = pd.DataFrame({
    "feature": importances.index,
    "importance": importances.values,
    "rank": range(1, len(importances) + 1),
    "is_dds": [f in dds_feature_cols for f in importances.index],
})
print("Top 15 fitur keseluruhan:")
print(ranking.head(15).to_string(index=False))
print()
print("Semua fitur DDS dan rankingnya:")
print(ranking[ranking["is_dds"]].to_string(index=False))


Top 15 fitur keseluruhan:
            feature  importance  rank  is_dds
     dd_par_denom_H    0.026221     1    True
     dd_par_denom_S    0.022526     2    True
       ns_has_fit_S    0.020229     3   False
       ew_has_fit_S    0.018564     4   False
       ns_has_fit_H    0.018170     5   False
     dd_par_denom_N    0.016788     6    True
       dd_par_level    0.016158     7    True
   hcp_ns_advantage    0.014585     8   False
opening_strain_PASS    0.014465     9   False
             ew_hcp    0.013728    10   False
           ns_fit_S    0.013634    11   False
             ns_hcp    0.013385    12   False
           ns_fit_H    0.012914    13   False
           ew_fit_S    0.012245    14   False
     dd_par_denom_D    0.011222    15    True

Semua fitur DDS dan rankingnya:
              feature  importance  rank  is_dds
       dd_par_denom_H    0.026221     1    True
       dd_par_denom_S    0.022526     2    True
       dd_par_denom_N    0.016788     6    True
         dd_p

## 5. Simpan hasil

In [6]:
comparison = compare_models(results)
comparison.to_csv(OUT_DIR / "val_comparison.csv")
ranking.to_csv(OUT_DIR / "feature_importance_ranking.csv", index=False)

summary = {
    "generated": pd.Timestamp.now().isoformat(),
    "dds_feature_cols": dds_feature_cols,
    "results": {r["model"]: {k: r[k] for k in
        ["accuracy", "f1_macro", "f1_weighted", "top_3_accuracy", "top_5_accuracy"]}
        for r in results},
    "dds_feature_ranks": ranking[ranking["is_dds"]].to_dict("records"),
}
(OUT_DIR / "summary.json").write_text(json.dumps(summary, indent=2))
print(f"Saved: {OUT_DIR / 'val_comparison.csv'}")
comparison


Saved: D:\Bridge-Prediction\experiments\2026-07-15\outputs\dds_features\val_comparison.csv


,accuracy,precision_macro,recall_macro,f1_macro,f1_weighted,top_3_accuracy,top_5_accuracy
model,,,,,,,
A: XGBoost baseline (164 fitur),0.512720,0.333505,0.251472,0.268524,0.468546,0.750815,0.848010
B: XGBoost +DDS (182 fitur),0.511416,0.337018,0.260010,0.278593,0.471787,0.751468,0.846053
C: XGBoost +DDS +sample_weight +dart (182 fitur),0.484671,0.310113,0.343259,0.315411,0.491443,0.736464,0.833007
"D (REF): XGBoost +sample_weight +dart, TANPA DDS (164 fitur)",0.491846,0.315552,0.345488,0.321666,0.498336,0.748206,0.833007


## 6. Kesimpulan

Hasil val set (`experiments/2026-07-15/outputs/dds_features/val_comparison.csv`):

| Model | Accuracy | F1 macro | F1 weighted |
|---|---|---|---|
| A: XGBoost baseline (164 fitur) | **51.3%** | 0.269 | 0.469 |
| B: XGBoost +DDS (182 fitur) | 51.1% | **0.279** | **0.472** |
| C: XGBoost +DDS+sample_weight+DART (182 fitur) | 48.5% | 0.315 | 0.491 |
| D (REF): XGBoost +sample_weight+DART, TANPA DDS (164 fitur) | 49.2% | **0.322** | **0.498** |

**DDS memberi sinyal yang GENUINE dan kuat** — beda dari fitur tambahan
2026-07-09 yang sebagian besar cuma kombinasi linear dari fitur lama.
`dd_par_denom_H` dan `dd_par_denom_S` (strain kontrak par) adalah
**fitur PALING PENTING dari SELURUH 182 fitur** (rank #1 dan #2,
mengalahkan `ns_has_fit_S` yang sebelumnya selalu di puncak).
`dd_par_denom_N`, `dd_par_level` juga masuk top-10. Ini masuk akal —
par contract secara langsung menjawab "strain apa yang optimal", jauh
lebih dekat ke jawaban dibanding proxy HCP/fit.

**Tapi pola yang sama seperti kemarin (2026-07-10) berulang**: menambah
DDS ke kombinasi yang SUDAH bagus (sample_weight+DART) JUSTRU
menurunkan hasil (F1 macro 0.322→0.315, F1 weighted 0.498→0.491) —
padahal DDS+XGBoost polos (B) sedikit lebih baik dari baseline polos
(A). **Konfirmasi kedua** untuk pola "perbaikan tidak aditif": teknik
yang membantu model sederhana tidak otomatis menambah nilai ke
kombinasi yang sudah dioptimalkan.

**Kenapa DDS tidak menembus 52% meski sinyalnya kuat**: ini
mengonfirmasi temuan notebook 01 hari ini — par contract adalah
kontrak **teoretis optimal**, tapi bidding manusia asli menyimpang dari
itu (gaya partnership, konvensi, keputusan under uncertainty). DDS
memberi informasi genuinely baru (bukan cuma proxy dari yang sudah
ada — terbukti dari ranking #1/#2), tapi tidak bisa mengalahkan
ambiguitas fundamental "target-nya sendiri tidak deterministik dari
kartu" yang sudah dibuktikan lewat analisis pasangan open/closed-room
(37.6% konsistensi manusia).

**Rekomendasi**:
1. Fitur DDS (`dd_par_denom_*`, `dd_par_level`, `ns/ew_dd_*`) layak
   dipertimbangkan masuk fitur kanonik kalau timnya OK dengan
   dependency `endplay` dan waktu precompute (~1 jam untuk 506 file,
   sekali saja, hasilnya di-cache) — beda dari fitur 2026-07-09 kemarin
   yang ranking-nya sekunder, DDS terbukti fitur PALING kuat.
2. Untuk kombinasi dengan DART/sample_weight: JANGAN — pola konsisten
   menunjukkan itu menurunkan hasil, bukan menambah.
3. Soal target awal (60% accuracy): DDS tidak mengubah kesimpulan hari
   ini — 51-52% tampaknya memang mendekati batas struktural untuk
   exact-contract 35-kelas, terlepas dari fitur secanggih apa pun,
   karena batasnya ada di data (ambiguitas bidding manusia), bukan di
   model atau fitur.
